# Day 07 — String Operations + Data Cleaning
**Estimated time:** 45–60 min
**Datasets:** `materials_inventory.csv`

## Learning Objectives
- Use the `.str` accessor for vectorized string operations
- Clean SAP text fields: leading zeros, whitespace, mixed case, regex extraction
- Work with ordered categoricals and apply custom cleaning functions

In [ ]:
import pandas as pd
import numpy as np
import re

from analytics_bootcamp.config import RAW_DATA_DIR as DATA_DIR
df = pd.read_csv(f"{DATA_DIR}/materials_inventory.csv")
df["LABST"] = pd.to_numeric(df["LABST"], errors="coerce")
df["STPRS"] = pd.to_numeric(df["STPRS"], errors="coerce")

print(df.shape)
display(df.head(8))

In [ ]:
# -- 1. .str accessor fundamentals --
# Vectorized string ops -- no loops needed
sample = pd.Series(["  Bearing Assembly  ", "GEAR BOX unit-12", "valve_TYPE_B ", "pump"])

print("upper:      ", sample.str.upper().tolist())
print("lower:      ", sample.str.lower().tolist())
print("strip:      ", sample.str.strip().tolist())
print("title:      ", sample.str.title().tolist())
print("len:        ", sample.str.len().tolist())
print("contains pump:", sample.str.contains("pump", case=False).tolist())
print("startswith v:", sample.str.startswith("valve").tolist())
print("replace _:  ", sample.str.replace("_", " ", regex=False).tolist())

In [ ]:
# -- 2. regex extraction from text fields --
# Extract numeric suffix from material descriptions
sample_series = pd.Series(["BEARING-01", "VALVE-999", "PUMP-7A", "GEAR BOX", "MOTOR 42"])
extracted = sample_series.str.extract(r"(\d+)")      # first capture group
print("Extracted numeric parts:")
print(pd.concat([sample_series.rename("raw"), extracted.rename({0:"number"})], axis=1))

In [ ]:
# -- 3. Clean MAKTX field --
# Raw issues: leading/trailing whitespace, mixed case, collapsed spaces
print("MAKTX sample (raw):")
print(df["MAKTX"].head(10).tolist())

df["MAKTX_CLEAN"] = (df["MAKTX"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)   # collapse internal spaces
)

print("\nMAKTX sample (cleaned):")
print(df["MAKTX_CLEAN"].head(10).tolist())
print(f"\nDistinct raw descriptions:   {df['MAKTX'].nunique()}")
print(f"Distinct clean descriptions: {df['MAKTX_CLEAN'].nunique()}")

In [ ]:
# -- 4. Leading-zero padding for SAP keys --
# SAP MATNR is 18 chars, WERKS is 4 chars
df["MATNR_PADDED"] = df["MATNR"].astype(str).str.strip().str.zfill(18)
df["WERKS_PADDED"] = df["WERKS"].astype(str).str.strip().str.zfill(4)

print("MATNR padded sample:", df["MATNR_PADDED"].head(5).tolist())
print("WERKS padded sample:", df["WERKS_PADDED"].head(5).tolist())

In [ ]:
# -- 5. pd.Categorical for ordered data --
# MRPTYPE is a coded field with known grouping
mrp_order = ["PD","MK","MN","ND","VM",""]
df["MRPTYPE_CAT"] = pd.Categorical(
    df["MRPTYPE"].astype(str).str.strip().str.upper(),
    categories=mrp_order,
    ordered=False
)

print("MRPTYPE value counts (as Category):")
print(df["MRPTYPE_CAT"].value_counts())
print(f"\nMemory (Category): {df['MRPTYPE_CAT'].nbytes} bytes vs object: {df['MRPTYPE'].nbytes} bytes")

In [ ]:
# -- 6. Apply custom cleaning functions --

def clean_sap_text(val) -> str:
    # Standardize SAP text field: strip, uppercase, collapse whitespace
    if pd.isna(val):
        return ""
    return " ".join(str(val).strip().upper().split())

def pad_matnr(val, width=18) -> str:
    # Zero-pad a material number to SAP standard length
    return str(val).strip().zfill(width)

# Apply via .map() (element-wise, no index alignment needed)
df["MAKTX_V2"] = df["MAKTX"].map(clean_sap_text)
df["MATNR_V2"] = df["MATNR"].map(pad_matnr)

# Verify equivalence
assert (df["MAKTX_V2"] == df["MAKTX_CLEAN"]).all(), "Mismatch in cleaning approaches"
print("Both cleaning approaches produce identical results")

In [ ]:
# -- 7. Detect duplicate descriptions mapping to multiple MATNRs --
desc_to_matnr = (df.groupby("MAKTX_CLEAN")["MATNR"]
                   .nunique()
                   .rename("distinct_matns"))

ambiguous = desc_to_matnr[desc_to_matnr > 1].sort_values(ascending=False)
print(f"Descriptions mapping to multiple MATNRs: {len(ambiguous)}")
display(ambiguous.head(10))

---
## Daily Prompt

**The `MAKTX` field has inconsistent casing and trailing spaces. Clean it and then count distinct material descriptions that map to multiple MATNRs.**

```python
# YOUR SOLUTION
def clean_maktx(val):
    # YOUR SOLUTION: strip, upper, collapse spaces
    pass

df["MAKTX_FINAL"] = df["MAKTX"].map(clean_maktx)

desc_matnr_map = (
    df.groupby("MAKTX_FINAL")["MATNR"]
    .agg(["nunique", lambda x: list(x.unique()[:3])])
)
# YOUR SOLUTION: filter for nunique > 1 and examine patterns
```

> **Hint:** After finding ambiguous descriptions, check if the duplicates are the same material in different plants (`WERKS`), or genuinely different materials with the same description — a data quality issue worth flagging to the MDM team.